# Bottom roughness and tidal velocity amplitude

ACCESS-OM3 parameterises mixing from breaking internal tides (`INT_TIDE_DISSIPATION = True`, St. Laurent et al. 2002 energy input with the `POLZIN_09` vertical profile). The energy converted from the barotropic tide into internal waves scales as

$$E \propto \kappa \, h^2 \, N_b \, U_{tide}^2 ,$$

so the two input fields that set where tidal mixing happens are
- **`h2`**: bottom roughness squared (m²), read from `H2_FILE`;
- **`tideamp`**: barotropic tidal velocity amplitude (m/s), read from `TIDEAMP_FILE` (from TPXO10).

This notebook maps the two fields as used by the run, following Figure A5 of [Griffies et al. (2024)](http://dx.doi.org/10.22541/essoar.173282145.53065190/v1) (CM4X), and compares them with the equivalent ACCESS-OM2-025 inputs.

The fields are model *inputs*, so they don't change with time; the notebook finds the files the run actually used from the run's `config.yaml`.

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
# e.g. https://access-om3-configs.access-hive.org.au/latest/Experiments/
# esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_era_iaf+wombatlite-test5.1-era-b3d64e3d/datastore.json"
# esm_file = "/g/data/ik11/outputs/access-om3-025/MC_25km_jra_ryf+wombatlite-test5.1-5c6ad028/datastore.json"
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test5.1-8f50b769/datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
if not papermill: 
    import nci_ipynb, os  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import os
import intake
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean as cm
from exptdata_access import get_exptname_from_path

# Stop annoying warnings coming out of xarray.
import warnings
warnings.filterwarnings('ignore')

## Find the roughness and tidal amplitude files used by the OM3 run

Every payu run directory (`outputNNN`) keeps a copy of `config.yaml`, whose `input:` list gives the full path of every input file. We look in the first output directory of the experiment for the files called `bottom_roughness.nc` (`H2_FILE`) and `tideamp.nc` (`TIDEAMP_FILE`), and also print the relevant MOM6 parameters so the figure is tied to the run's settings.

In [ ]:
# The datastore sits in the experiment directory, next to output000/, output001/, ...
experiment_dir = os.path.dirname(esm_file)
config_file = os.path.join(experiment_dir, "output000", "config.yaml")
parameter_file = os.path.join(experiment_dir, "output000", "MOM_parameter_doc.all")

roughness_file_OM3 = None
tideamp_file_OM3 = None
with open(config_file) as f:
    for line in f:
        line = line.strip().lstrip("- ").strip()
        if line.endswith("/bottom_roughness.nc"):
            roughness_file_OM3 = line
        if line.endswith("/tideamp.nc"):
            tideamp_file_OM3 = line

print("Roughness file:", roughness_file_OM3)
print("Tidal amplitude file:", tideamp_file_OM3)

# Show the internal-tide mixing settings of this run
with open(parameter_file) as f:
    for line in f:
        name = line.split("=")[0].strip()
        if name in ["INT_TIDE_DISSIPATION", "INT_TIDE_PROFILE", "INT_TIDE_DECAY_SCALE", "KAPPA_ITIDES",
                    "KAPPA_H2_FACTOR", "H2_FILE", "TIDEAMP_FILE", "UTIDE", "MU_ITIDES", "GAMMA_ITIDES"]:
            print(line.split("!")[0].rstrip())

In [ ]:
# Bottom roughness squared, h^2 (m^2), and tidal velocity amplitude (m/s), on the OM3 grid (dims y/x and yh/xh)
h2_OM3 = xr.open_dataset(roughness_file_OM3).h2
tideamp_OM3 = xr.open_dataset(tideamp_file_OM3).tideamp.squeeze("time", drop=True)

for da in [h2_OM3, tideamp_OM3]:
    print(da.name, "|", da.attrs.get("long_name"), "|", da.attrs.get("units"))

# Ocean mask from the model bathymetry (deptho > 0 is ocean)
deptho_search = intake.open_esm_datastore(esm_file, columns_with_iterables=["variable"]).search(variable="deptho")
deptho_OM3 = deptho_search.search(path=deptho_search.df.path[0]).to_dask()["deptho"].load()
ocean_OM3 = deptho_OM3.values > 0   # numpy array (yh, xh); the input files use different dimension names

# Mask land and take log10 of h^2 (h^2 = 0 would give -inf, so leave those cells out)
h2_OM3 = h2_OM3.where(ocean_OM3 & (h2_OM3.values > 0))
log10_h2_OM3 = np.log10(h2_OM3)
tideamp_OM3 = tideamp_OM3.where(ocean_OM3)

lon_OM3 = h2_OM3.lon   # 2D true longitude/latitude (no NaNs), so the tripolar Arctic is drawn correctly
lat_OM3 = h2_OM3.lat
om3_res = {360: "100km", 1440: "25km", 4320: "8km"}[deptho_OM3.sizes["xh"]]

## ACCESS-OM2-025 inputs for comparison

ACCESS-OM2 (MOM5) uses the Simmons et al. (2004) tidal mixing scheme with `roughness_amp.nc` (roughness **amplitude** h, in m, from 20–100 km band-pass filtered topography) and `tideamp.nc`. We square the roughness amplitude so it can be compared with OM3's h². Note that the two roughness products are built differently (OM3: regridded from a WOA-based high-resolution topography product with the om3-scripts tools; OM2: 2012 GFDL CM2.5 product), so differences reflect the input datasets as well as resolution.

The OM2-025 `tideamp.nc` (created 2010, no provenance metadata in the file) looks much smoother than OM3's and has blob-like maxima around small Pacific islands; this is a property of that input file (it appears the same on the native OM2 grid), most likely from a coarser tidal product interpolated to 0.25°. OM3's `tideamp.nc` is regridded conservatively from TPXO10-atlas v2 (1/30°).

In [ ]:
OM2_input_dir = "/g/data/ik11/inputs/access-om2/input_20201102/mom_025deg"

roughness_OM2 = xr.open_dataset(f"{OM2_input_dir}/roughness_amp.nc")
tideamp_OM2_file = xr.open_dataset(f"{OM2_input_dir}/tideamp.nc")
print("roughness_amp |", roughness_OM2.roughness_amp.attrs.get("long_name"))

ocean_OM2 = roughness_OM2.wet > 0   # dims (ydim, xdim)

h2_OM2 = (roughness_OM2.roughness_amp ** 2).where(ocean_OM2 & (roughness_OM2.roughness_amp > 0))
log10_h2_OM2 = np.log10(h2_OM2)
tideamp_OM2 = tideamp_OM2_file.tideamp.where(ocean_OM2)

# The alon/alat stored in these old input files are wrong in the tripolar Arctic (top row sits at ~67°N),
# which smears cells across the land. Use the true T-cell centres from the OM2-025 supergrid instead:
# ocean_hgrid.nc holds a grid at twice the resolution, so the cell centres are the odd-indexed points.
hgrid_OM2 = xr.open_dataset(f"{OM2_input_dir}/ocean_hgrid.nc")
lon_OM2 = hgrid_OM2.x[1::2, 1::2].values   # (1080, 1440), same shape as the roughness/tide fields
lat_OM2 = hgrid_OM2.y[1::2, 1::2].values

### Put the OM2-025 fields on the OM3 grid

To difference the two models, the OM2-025 fields are mapped onto the OM3-25km grid by nearest neighbour (the grids have similar resolution, so this is enough for a map). Each grid point is turned into a 3D position on the unit sphere so distances are correct everywhere, including near the poles and across the date line. Only points that are ocean in both models are kept.

In [ ]:
from scipy.spatial import cKDTree

# Longitude/latitude (degrees) -> x, y, z on the unit sphere, for every OM2 and OM3 grid point
lon_OM2_rad, lat_OM2_rad = np.deg2rad(lon_OM2), np.deg2rad(lat_OM2)
xyz_OM2 = np.stack([np.cos(lat_OM2_rad) * np.cos(lon_OM2_rad),
                    np.cos(lat_OM2_rad) * np.sin(lon_OM2_rad),
                    np.sin(lat_OM2_rad)], axis=-1).reshape(-1, 3)        # (1080*1440, 3)

lon_OM3_rad, lat_OM3_rad = np.deg2rad(lon_OM3.values), np.deg2rad(lat_OM3.values)
xyz_OM3 = np.stack([np.cos(lat_OM3_rad) * np.cos(lon_OM3_rad),
                    np.cos(lat_OM3_rad) * np.sin(lon_OM3_rad),
                    np.sin(lat_OM3_rad)], axis=-1).reshape(-1, 3)        # (1152*1440, 3)

# For every OM3 point, the index of the closest OM2 point
_, nearest_OM2_index = cKDTree(xyz_OM2).query(xyz_OM3)

# Pick the OM2 values at those points and reshape to the OM3 grid (y, x), keeping OM3 dimension names
log10_h2_OM2_on_OM3 = xr.DataArray(log10_h2_OM2.values.ravel()[nearest_OM2_index].reshape(log10_h2_OM3.shape),
                                   dims=log10_h2_OM3.dims).where(ocean_OM3)
tideamp_OM2_on_OM3 = xr.DataArray(tideamp_OM2.values.ravel()[nearest_OM2_index].reshape(tideamp_OM3.shape),
                                  dims=tideamp_OM3.dims).where(ocean_OM3)

# Differences OM3 - OM2. For roughness, the difference of log10 h^2 is log10 of the ratio h^2_OM3 / h^2_OM2.
log10_h2_ratio = log10_h2_OM3 - log10_h2_OM2_on_OM3.values
tideamp_difference = tideamp_OM3 - tideamp_OM2_on_OM3.values

## Plotting

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12), subplot_kw={"projection": ccrs.Robinson(central_longitude=-150)})
ft_size = 12

log10_h2_limits = (2, 6)    # h^2 from 100 m^2 (h = 10 m) to 10^6 m^2 (h = 1 km)
tideamp_limits = (0, 0.1)   # m/s; shelves reach > 1 m/s, so the colour bar saturates there

panels = [
    # (row, col, field, lon, lat, cmap, vmin, vmax, colour-bar label, title)
    (0, 0, log10_h2_OM3, lon_OM3, lat_OM3, cm.cm.amp, *log10_h2_limits, "log$_{10}$ h$^2$ (m$^2$)", f"ACCESS-OM3-{om3_res}: bottom roughness"),
    (0, 1, tideamp_OM3,  lon_OM3, lat_OM3, cm.cm.speed, *tideamp_limits, "U$_{tide}$ (m s$^{-1}$)", f"ACCESS-OM3-{om3_res}: tidal velocity amplitude"),
    # OM2-025 is drawn from its values mapped onto the OM3 grid (see above), so both rows use the same coordinates
    (1, 0, log10_h2_OM2_on_OM3, lon_OM3, lat_OM3, cm.cm.amp, *log10_h2_limits, "log$_{10}$ h$^2$ (m$^2$)", "ACCESS-OM2-025: bottom roughness"),
    (1, 1, tideamp_OM2_on_OM3,  lon_OM3, lat_OM3, cm.cm.speed, *tideamp_limits, "U$_{tide}$ (m s$^{-1}$)", "ACCESS-OM2-025: tidal velocity amplitude"),
    (2, 0, log10_h2_ratio,      lon_OM3, lat_OM3, cm.cm.balance, -2, 2, "log$_{10}$(h$^2_{OM3}$ / h$^2_{OM2}$)", "Roughness: OM3 − OM2 (log ratio)"),
    (2, 1, tideamp_difference,  lon_OM3, lat_OM3, cm.cm.balance, -0.03, 0.03, "ΔU$_{tide}$ (m s$^{-1}$)", "Tidal velocity amplitude: OM3 − OM2"),
]

panel_letters = "abcdef"
for i, (row, col, field, lon, lat, cmap, vmin, vmax, label, title) in enumerate(panels):
    ax = axes[row, col]
    ax.set_global()
    ax.add_feature(cfeature.LAND, color="darkgrey", zorder=0)
    # pcolormesh with 2D coordinates; the cells that straddle the map edge are dropped by cartopy
    im = ax.pcolormesh(lon, lat, field, cmap=cmap, vmin=vmin, vmax=vmax, transform=ccrs.PlateCarree(), rasterized=True)
    plt.colorbar(im, ax=ax, shrink=0.7, extend="both", label=label)
    ax.set_title(title, fontsize=ft_size)
    ax.set_title(panel_letters[i], fontweight="bold", fontsize=ft_size, loc="left")

fig.suptitle(get_exptname_from_path(esm_file), fontsize=ft_size)
fig.tight_layout()

mkmd.savefig(fig, "Bottom roughness and tidal velocity",
             f"Inputs to the internal-tide mixing parameterisation (cf. Griffies et al. 2024, Fig. A5). "
             f"(a) log10 of bottom roughness squared h^2 (m^2) and (b) barotropic tidal velocity amplitude (m/s, TPXO10) for ACCESS-OM3-{om3_res}; "
             f"(c,d) the same for ACCESS-OM2-025 (roughness amplitude squared; Simmons et al. 2004 scheme inputs), mapped to the OM3 grid by nearest neighbour; "
             f"(e) log10 of the ratio OM3/OM2 of h^2 and (f) OM3 minus OM2 tidal velocity amplitude.", dpi=150)

In [ ]:
# Summary numbers: area-weighted global ocean medians/means are not very meaningful for such skewed fields,
# so print simple percentiles over ocean points for each model
import pandas as pd
rows = {}
for name, field in [("OM3 h (m)", np.sqrt(h2_OM3)), ("OM2-025 h (m)", np.sqrt(h2_OM2)),
                    ("OM3 U_tide (m/s)", tideamp_OM3), ("OM2-025 U_tide (m/s)", tideamp_OM2)]:
    values = field.values[np.isfinite(field.values)]
    rows[name] = {"10th pct": np.percentile(values, 10), "median": np.median(values), "90th pct": np.percentile(values, 90), "max": values.max()}
pd.DataFrame(rows).T.round(3)